In [1]:
#그래프관련 import
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
# matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus']=False

import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso,LogisticRegression
from sklearn.preprocessing import StandardScaler,PolynomialFeatures,OneHotEncoder     
from sklearn.metrics import mean_absolute_error
from scipy.special import expit



In [2]:
# 데이터 불러오기
fish = pd.read_csv('data/fish_logistic.csv')
fish.head(2)
fish['Species'].unique() # 7개의 특성(종류)을 가지고 있음.

<StringArray>
['Bream', 'Roach', 'Whitefish', 'Parkki', 'Perch', 'Pike', 'Smelt']
Length: 7, dtype: str

In [3]:
# fish.info()
# fish.describe()

In [4]:
fish.columns

Index(['Species', 'Weight', 'Length', 'Diagonal', 'Height', 'Width'], dtype='str')

In [5]:
# 데이터 전처리
data = fish[fish.columns[1:]].values
label = fish['Species']

In [6]:
# 학습데이터,테스트 데이너 분리
train_data,test_data,train_label,test_label = train_test_split(data,label,random_state=42)

In [7]:
# LogisticRegression,KNN 은 무조건 정규화 작업을 해야 함.
ss = StandardScaler()
ss.fit(train_data)
train_scaled = ss.transform(train_data)
test_scaled = ss.transform(test_data)


In [8]:
### 이진분류 형태 구성
### Bream,Smelt 생선만 분리
bream_smelt_index = (train_label=='Bream') | (train_label=='Smelt')
train_bream_smelt = train_scaled[bream_smelt_index]
label_bream_smelt = train_label[bream_smelt_index]
label_bream_smelt.shape
bream_smelt_index.value_counts()

Species
False    86
True     33
Name: count, dtype: int64

In [9]:
# 데이터 학습
lr = LogisticRegression()
lr.fit(train_bream_smelt,label_bream_smelt)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [10]:
lr.classes_

array(['Bream', 'Smelt'], dtype=object)

In [11]:
print('분류 : ',lr.predict(train_bream_smelt[:5]))
#확률을 보여줌
print(lr.classes_)
print(lr.predict_proba(train_bream_smelt[:5]))

# 데이터 예측
# 정확도 

분류 :  ['Bream' 'Smelt' 'Bream' 'Bream' 'Bream']
['Bream' 'Smelt']
[[0.99760007 0.00239993]
 [0.02737325 0.97262675]
 [0.99486386 0.00513614]
 [0.98585047 0.01414953]
 [0.99767419 0.00232581]]


In [12]:
fish.columns

Index(['Species', 'Weight', 'Length', 'Diagonal', 'Height', 'Width'], dtype='str')

In [13]:
in_data = train_data[bream_smelt_index]
in_data[:5]

array([[720.    ,  35.    ,  40.6   ,  16.3618,   6.09  ],
       [  7.5   ,  10.5   ,  11.6   ,   1.972 ,   1.16  ],
       [620.    ,  34.5   ,  39.7   ,  15.5227,   5.2801],
       [500.    ,  31.    ,  36.2   ,  14.3714,   4.8146],
       [714.    ,  36.    ,  41.5   ,  16.517 ,   5.8515]])

In [ ]:
# 'Weight', 'Length', 'Diagonal', 'Height', 'Width'
# 720,35,40.6,16.3618,6.09
# 7.5,10.5,11.6,1.972,1.16
# 생선의 무게,길이,대각선,높이,넓이를 입력받아, 어떤 생선인지 분류확률을 출력하시오.
# 분류생선확률 : Bream (99.76%)

In [23]:
in_data = '720,35,40.6,16.3618,6.09'
in_data = in_data.split(',')
in_data = list(map(float,in_data))
in_data

[720.0, 35.0, 40.6, 16.3618, 6.09]

In [24]:
# # 3. 정규화
in_data_scaled = ss.transform([in_data])

# 예측
result = lr.predict(in_data_scaled)
result2 = lr.predict_proba(in_data_scaled)
print(result2)
if result[0] == 'Bream':
    result3 = result2[0][0]
else:
    result3 = result2[0][1]
print("test : ",result3)
print(f"분류생선확률 : {result[0]} ( {result3*100:.2f}% )")
# 분류생선확률 : Bream (99.76%)

[[0.99760007 0.00239993]]
test :  0.9976000716505207
분류생선확률 : Bream ( 99.76% )


In [25]:
print(lr.coef_,lr.intercept_) # logisticRegression 선그래프 표현가능

[[-0.40451732 -0.57582787 -0.66248158 -1.01329614 -0.73123131]] [-2.16172774]


In [26]:
# z의 값을 출력하는 함수
# z = a*무게 + b*길이 + c*대각선 + d*높이 + e*두께 + f : decision_function
decsions = lr.decision_function(train_bream_smelt[:5])
decsions

array([-6.02991358,  3.57043428, -5.26630496, -4.24382314, -6.06135688])

In [27]:
# 이진분류에 사용되는 로직스틱 함수 ( 0-1까지 값으로 변경해서 보여줌 )
from scipy.special import expit
print(expit(decsions))

[0.00239993 0.97262675 0.00513614 0.01414953 0.00232581]


In [28]:
label_bream_smelt[:5]

26     Bream
146    Smelt
22     Bream
11     Bream
27     Bream
Name: Species, dtype: str